# Phase 3 — Adapters, Registry, Caching, Retry

> Run top-to-bottom AFTER Phase 3 is built (restart the kernel if you pulled new code). No Gemini calls in this notebook — it exercises the resilience layer only.

**Goal:** see the tool registry, the adapter fallback chain, a real cache hit, and a clean error.


In [1]:
# --- Setup ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env loaded OK')

env loaded OK


## 1. Registered tools grouped by agent (Registry pattern)
Tools declare themselves with `@tool_registry.register(agent=...)`; agents discover their toolbox — no hand-maintained lists.

In [2]:
from app.tools.registry import tool_registry
from app.tools import portfolio_tools, market_tools  # importing triggers registration

for row in tool_registry.table():
    print(f"{row['agent']:16} {row['tool']:30} {row['description'][:55]}")

market_research  get_market_snapshot            Current price, 1-day / 1-month change, 52-week range, a
market_research  get_recent_news                Recent news headlines for a symbol. Says so honestly wh
market_research  get_sector_performance         1-day and 5-day performance of the 11 S&P sectors (via 
market_research  get_economic_indicators        Macro indicators (rates, inflation, unemployment). STUB
portfolio        get_holdings                   List every holding for a client: symbol, security_name,
portfolio        get_portfolio_value            Total portfolio market value, marked to market. Cash is
portfolio        get_position                   One position's detail: quantity, cost basis, current pr
portfolio        get_position_performance       Total return of one position since purchase (current pr
portfolio        get_ytd_returns                Year-to-date return per holding (vs the first trading d
portfolio        get_allocation_by_sector       Portfolio alloca

## 2. The adapter fallback chain (Chain of Responsibility)
Keyed vendors first, keyless yfinance last. An adapter with no API key reports itself unavailable and is skipped — with zero optional keys this still works.

In [3]:
from app.integrations.chain import market_data
for s in market_data.sources():
    print(f"{s['adapter']:15} available={s['available']}")

quote = market_data.get_quote('NVDA')
print('\nNVDA quote:', quote['price'], '| served by:', quote['source'])

finnhub         available=True
alpha_vantage   available=True
yfinance        available=True
{"method": "get_quote", "source": "finnhub", "event": "chain_served", "level": "info", "timestamp": "2026-07-12T19:40:12.418419Z"}

NVDA quote: 210.96 | served by: finnhub


## 3. Cache hit — second identical call costs ~nothing (Decorator pattern)
`@cached` writes to `.cache/` on disk, so even a fresh kernel benefits.

In [ ]:
import time
from app.tools.market_tools import get_market_snapshot

t0 = time.perf_counter(); a = get_market_snapshot('MSFT'); t1 = time.perf_counter()
b = get_market_snapshot('MSFT'); t2 = time.perf_counter()
print(f'first call : {t1-t0:6.2f}s  (network)')
print(f'second call: {t2-t1:6.4f}s  (cache hit — same result: {a["price"] == b["price"]})')

{"method": "get_price_history", "adapter": "finnhub", "error": "[finnhub] price history requires a paid plan", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T19:42:10.587003Z"}
{"method": "get_price_history", "adapter": "alpha_vantage", "error": "[alpha_vantage] period '1y' unsupported on free tier", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T19:42:10.587706Z"}
{"method": "get_price_history", "source": "yfinance", "event": "chain_served", "level": "info", "timestamp": "2026-07-12T19:42:11.100428Z"}
{"fn": "get_market_snapshot", "age_s": 0.0, "event": "cache_hit", "level": "info", "timestamp": "2026-07-12T19:42:11.101378Z"}
first call :   0.51s  (network)
second call: 0.0005s  (cache hit — same result: True)


## 4. A bad ticker fails CLEANLY — one ToolError, no stack trace

In [5]:
from app.errors.exceptions import ToolError
try:
    market_data.get_quote('NOT_A_REAL_TICKER_XYZ')
except ToolError as e:
    print('Clean ToolError:', str(e)[:200])

{"method": "get_quote", "adapter": "finnhub", "error": "[finnhub] no quote for 'NOT_A_REAL_TICKER_XYZ'", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T19:43:08.037168Z"}
{"method": "get_quote", "adapter": "alpha_vantage", "error": "[alpha_vantage] no quote for 'NOT_A_REAL_TICKER_XYZ'", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T19:43:08.649971Z"}


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOT_A_REAL_TICKER_XYZ"}}}
$NOT_A_REAL_TICKER_XYZ: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$NOT_A_REAL_TICKER_XYZ: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


{"method": "get_quote", "adapter": "yfinance", "error": "'exchangeTimezoneName'", "event": "chain_adapter_failed", "level": "warning", "timestamp": "2026-07-12T19:43:10.980240Z"}
Clean ToolError: All market-data sources failed for get_quote('NOT_A_REAL_TICKER_XYZ',): finnhub: [finnhub] no quote for 'NOT_A_REAL_TICKER_XYZ'; alpha_vantage: [alpha_vantage] no quote for 'NOT_A_REAL_TICKER_XYZ'; yf


## 5. Bonus: Gemini itself is rate-limited now
The LLM factory attaches an `InMemoryRateLimiter` (~8 req/min) so bursts queue briefly instead of eating a 429 from the free tier.

In [6]:
from app.llm.factory import get_llm
print('Gemini rate limiter attached:', get_llm().rate_limiter is not None)

Gemini rate limiter attached: True


## ✅ Acceptance check

In [7]:
assert len(tool_registry.tools_for('portfolio')) == 8
assert len(tool_registry.tools_for('market_research')) == 4
assert quote['price'] > 0
assert (t2-t1) < (t1-t0)  # cached beats network
print('Phase 3 acceptance: PASS')

Phase 3 acceptance: PASS
